In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt

import numpy as np

from tqdm import tqdm
import random
import torch
import torchvision

from PIL import Image
from torchvision.models import AlexNet, AlexNet_Weights
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import vgg16, VGG16_Weights

import scipy.io

from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors

import math

from pathlib import Path

# Fixa a semente para o Python e Numpy
SEED = 692
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms
from typing import Any, Callable


class Flowers(Dataset):
    def __init__(
        self,
        root_dir: str | Path,
        split: str = "val",
        transform: Callable[[Any], torch.Tensor] | None = None,
        target_transform: Callable[[Any], torch.Tensor] | None = None,
    ):
        super().__init__()

        self.root_dir = Path(root_dir)
        self.transform = transform
        self.target_transform = target_transform

        split = split.lower()
        assert split in {"train", "val", "test"}

        mat = scipy.io.loadmat(self.root_dir / "datasplits.mat")

        if split == "train":
            ids = mat["trn1"][0]
        elif split == "val":
            ids = mat["val1"][0]
        else:
            ids = mat["tst1"][0]

        self.samples = []

        for image_id in ids:
            image_id = int(image_id)

            label = (image_id - 1) // 80

            image_path = (
                self.root_dir
                / "jpg"
                / str(label)
                / f"image_{image_id:04d}.jpg"
            )

            self.samples.append((image_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        image_path, label = self.samples[idx]

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        if self.target_transform is not None:
            label = self.target_transform(label)

        return image, label
    

default_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])
default_target_transforms = transforms.Compose(
    lambda x: torch.tensor(x)
)
root_path = "./datasets/flowers102/"
train_dataset = Flowers(
    root_path,
    split="train",
    transform=default_transforms
)
val_dataset = Flowers(
    root_path,
    split="val",
    transform=default_transforms
)
test_dataset = Flowers(
    root_path,
    split="test",
    transform=default_transforms
)

len(train_dataset), len(val_dataset), len(test_dataset)

In [ ]:
sample_idx = np.random.randint(len(train_dataset))

def unitary_normalize(
    image: torch.Tensor
) -> torch.Tensor:
    mn = torch.amin(image, (1, 2))
    mx = torch.amax(image, (1, 2))
    norm = (image - mn[:, None, None]) / (mx[:, None, None] - mn[:, None, None])
    return norm

image, label = train_dataset[sample_idx]

image_np = unitary_normalize(image).permute(1, 2, 0).numpy()
plt.imshow(image_np)
plt.title(f"Train sample #{sample_idx}, label: {label}")

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_dataloader = DataLoader(
    train_dataset,
    BATCH_SIZE,
    shuffle=True,
    num_workers=2
)
val_dataloader = DataLoader(
    val_dataset,
    BATCH_SIZE,
    num_workers=2
)
test_dataloader = DataLoader(
    val_dataset,
    BATCH_SIZE,
    num_workers=2
)

In [ ]:
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights
from torchsummary import summary


class Resnet18(nn.Module):
    def __init__(
        self,
        num_classes: int,
        pretrained: bool = False,
        freeze_layers: tuple[str, ...] = (),
    ):
        super().__init__()

        self.backbone = resnet18(
            weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None,
        )

        for name, module in self.backbone.named_children():
            if name in freeze_layers:
                for param in module.parameters():
                    param.requires_grad = False

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)
    

model = Resnet18(
    num_classes=17
).to(device)

summary(model, input_size=(3, 224, 224), batch_size=BATCH_SIZE)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.optim import Adam
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm


def train(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs,
    early_stopper,
    device=None,
    checkpoint_dir="checkpoints",
    log_dir="runs",
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    writer = SummaryWriter(log_dir=log_dir)

    best_loss = float("inf")

    epoch_bar = tqdm(range(1, epochs + 1), desc="Training")
    for epoch in epoch_bar:
        metrics = {}

        #######################################################################
        # TRAIN
        #######################################################################
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        total_train_batches = len(train_loader)
        for batch_idx, (images, labels) in enumerate(train_loader, start=1):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            preds = logits.argmax(dim=1)

            train_loss += loss.item() * images.size(0)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

            metrics["batch"] = f"{batch_idx}/{total_train_batches}"
            metrics["train_loss"] = f"{train_loss/train_total:.4f}"
            metrics["train_acc"] = f"{100*train_correct/train_total:.2f}%"

            epoch_bar.set_postfix(metrics)

        train_loss /= train_total
        train_acc = train_correct / train_total

        #######################################################################
        # VALIDATION
        #######################################################################
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        y_true = []
        y_pred = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                logits = model(images)
                loss = criterion(logits, labels)

                preds = logits.argmax(dim=1)

                val_loss += loss.item() * images.size(0)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        val_loss /= val_total
        val_acc = val_correct / val_total

        #######################################################################
        # Atualiza métricas do postfix
        #######################################################################
        metrics["train_loss"] = f"{train_loss:.4f}"
        metrics["train_acc"] = f"{100*train_acc:.2f}%"
        metrics["val_loss"] = f"{val_loss:.4f}"
        metrics["val_acc"] = f"{100*val_acc:.2f}%"

        epoch_bar.set_postfix(metrics)

        #######################################################################
        # TensorBoard
        #######################################################################
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Accuracy/train", train_acc, epoch)
        writer.add_scalar("Accuracy/val", val_acc, epoch)

        #######################################################################
        # Confusion Matrix
        #######################################################################
        cm = confusion_matrix(
            y_true,
            y_pred,
            normalize="true",
        )

        fig, ax = plt.subplots(figsize=(8, 8))

        im = ax.imshow(cm, cmap="Blues", vmin=0.0, vmax=1.0)
        plt.colorbar(im, ax=ax)

        ax.set_title(f"Validation Confusion Matrix - Epoch {epoch}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")

        threshold = 0.5

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(
                    j,
                    i,
                    f"{cm[i, j]:.2f}",
                    ha="center",
                    va="center",
                    color="white" if cm[i, j] > threshold else "black",
                    fontsize=8,
                )

        ax.set_xticks(range(cm.shape[1]))
        ax.set_yticks(range(cm.shape[0]))
        plt.tight_layout()

        writer.add_figure(
            "Validation/Confusion Matrix",
            fig,
            global_step=epoch,
        )

        plt.close(fig)

        #######################################################################
        # Checkpoints
        #######################################################################
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }

        torch.save(checkpoint, checkpoint_dir / "last.pt")

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(checkpoint, checkpoint_dir / "best.pt")

        #######################################################################
        # Early stopping
        #######################################################################
        if early_stopper(val_loss):
            print(f"\nEarly stopping na época {epoch}.")
            break

    writer.close()


class EarlyStopping:
    def __init__(
        self,
        patience: int = 10,
        min_delta: float = 0.0,
    ):
        self.patience = patience
        self.min_delta = min_delta

        self.best_loss = float("inf")
        self.counter = 0

    def __call__(self, val_loss: float) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1

        return self.counter >= self.patience

    def reset(self):
        self.best_loss = float("inf")
        self.counter = 0


def test(
    model,
    test_loader,
    criterion,
    checkpoint_path: str | Path | None = None,
    device=None,
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    if checkpoint_path is not None:
        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True,
        )
        model.load_state_dict(checkpoint["model_state_dict"])

    model.eval()

    loss = 0.0

    y_true = []
    y_pred = []

    progress = tqdm(
        test_loader,
        desc="Testing",
        unit="batch",
    )

    with torch.no_grad():
        for images, labels in progress:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            batch_loss = criterion(logits, labels)

            preds = logits.argmax(dim=1)
            loss += batch_loss.item() * images.size(0)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

            metrics = {
                "loss": f"{loss / len(y_true):.4f}",
                "acc": f"{100 * accuracy_score(y_true, y_pred):.2f}%"
            }

            progress.set_postfix(metrics)

    loss /= len(test_loader.dataset)

    metrics = {
        "loss": loss,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
    }

    ###########################################################################
    # Confusion Matrix
    ###########################################################################
    cm = confusion_matrix(
        y_true,
        y_pred,
        normalize="true",
    )

    fig, ax = plt.subplots(figsize=(8, 8))

    im = ax.imshow(cm, cmap="Blues", vmin=0.0, vmax=1.0)
    plt.colorbar(im, ax=ax)

    ax.set_title(f"Validation Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    threshold = 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                f"{cm[i, j]:.2f}",
                ha="center",
                va="center",
                color="white" if cm[i, j] > threshold else "black",
                fontsize=8,
            )

    ax.set_xticks(range(cm.shape[1]))
    ax.set_yticks(range(cm.shape[0]))

    plt.tight_layout()

    ###########################################################################
    # Print metrics
    ###########################################################################
    print("\nTest Results")
    print("-" * 40)

    for key, value in metrics.items():
        if key == "loss":
            print(f"{key:<10}: {value:.4f}")
        else:
            print(f"{key:<10}: {100 * value:.2f}%")

    return metrics, fig

### No pretrain model

In [ ]:
model = Resnet18(
    num_classes=17,
    pretrained=False,
)

train(
    model,
    train_dataloader,
    criterion=nn.CrossEntropyLoss(),
    optimizer=Adam(model.parameters(), lr=1e-4),
    epochs=50,
    early_stopper=EarlyStopping(
        patience=5,
        min_delta=1e-4,
    ),
    val_loader=val_dataloader,
    checkpoint_dir="checkpoints/runs/resnet18-nopretrain",
    log_dir="tensorboard/runs/resnet18-nopretrain"
)

In [ ]:
metrics, fig = test(
    model,
    test_dataloader,
    nn.CrossEntropyLoss(),
    checkpoint_path="checkpoints/runs/resnet18-nopretrain/best.pt",
)

fig.show()

### Pre-trained model

In [ ]:
model = Resnet18(
    num_classes=17,
    pretrained=True,
    freeze_layers=(
        "conv1",
        "bn1",
        "layer1",
        "layer2",
        "layer3",
    )
)

train(
    model,
    train_dataloader,
    criterion=nn.CrossEntropyLoss(),
    optimizer=Adam(model.parameters(), lr=1e-4),
    epochs=50,
    early_stopper=EarlyStopping(
        patience=15,
        min_delta=1e-4,
    ),
    val_loader=val_dataloader,
    checkpoint_dir="checkpoints/runs/resnet18-pretrain",
    log_dir="tensorboard/runs/resnet18-pretrain"
)

In [ ]:
metrics, fig = test(
    model,
    test_dataloader,
    nn.CrossEntropyLoss(),
    checkpoint_path="checkpoints/runs/resnet18-pretrain/best.pt",
)

fig.show()

In [ ]:
import torch.nn.functional as F


def build_descriptors(
    model,
    dataset,
    batch_size=64,
    num_workers=0,
    device=None,
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()

    descriptor_model = nn.Sequential(
        *list(model.backbone.children())[:-1]
    ).to(device)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    descriptors = []
    labels = []

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Extracting descriptors"):
            images = images.to(device)

            feat = descriptor_model(images)
            feat = torch.flatten(feat, 1)
            feat = F.normalize(feat, p=2, dim=1)

            descriptors.append(feat.cpu())
            labels.append(targets)

    descriptors = torch.cat(descriptors)
    labels = torch.cat(labels)

    return descriptors, labels


def image_retrieval(
    descriptors,
    labels,
    query_index,
    k=5,
):
    similarity = descriptors @ descriptors[query_index]
    scores, indices = torch.topk(similarity, k + 1)
    return indices, scores, labels


def unitary_normalize(
    image: torch.Tensor
) -> torch.Tensor:
    mn = torch.amin(image, (1, 2))
    mx = torch.amax(image, (1, 2))
    norm = (image - mn[:, None, None]) / (mx[:, None, None] - mn[:, None, None])
    return norm


def plot_retrieval(
    dataset,
    query_index,
    retrieved_indices,
    scores=None,
):
    cols = 4
    rows = 1 + math.ceil(len(retrieved_indices) / cols)

    fig = plt.figure(figsize=(3 * cols, 3 * rows))
    gs = fig.add_gridspec(rows, cols)

    query_img, query_label = dataset[query_index]
    query_img = unitary_normalize(query_img)

    ax = fig.add_subplot(gs[0, 1])
    ax.imshow(query_img.permute(1, 2, 0))
    ax.set_title(f"Query\n{query_label}", fontsize=12, fontweight="bold")
    ax.axis("off")

    for i, idx in enumerate(retrieved_indices):
        row = 1 + i // cols
        col = i % cols

        img, label = dataset[int(idx)]
        img = unitary_normalize(img)

        ax = fig.add_subplot(gs[row, col])

        ax.imshow(img.permute(1, 2, 0))

        if scores is None:
            title = f"{label}"
        else:
            title = f"{label}\n{scores[i]:.3f}"

        ax.set_title(title, fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
checkpoint = torch.load(
    "./checkpoints/runs/resnet18-pretrain/best.pt",
    map_location=device,
    weights_only=True,
)
model.load_state_dict(checkpoint["model_state_dict"])

descriptors, labels = build_descriptors(
    model,
    test_dataset,
)

In [ ]:
#query_index = np.random.randint(len(test_dataloader))
query_index = 132
indices, scores, labels = image_retrieval(
    descriptors=descriptors,
    labels=labels,
    query_index=query_index,
    k=7,
)

plot_retrieval(
    dataset=test_dataset,
    query_index=query_index,
    retrieved_indices=indices,
    scores=scores,
)

| Método                      | Tipo              | Accuracy (%) | Configuração                          |
| :-------------------------- | :---------------- | -----------: | :------------------------------------ |
| LBP                         | Handcrafted       |         7.41 | 256 bins, dicionário 100              |
| SIFT-RNG                    | Handcrafted       |         7.68 | 128-D, dicionário 100                 |
| ORB-RNG                     | Handcrafted       |         6.85 | 32-D, dicionário 100                  |
| SIFT-Grid                   | Handcrafted       |         8.82 | 128-D, dicionário 100                 |
| ORB-Grid                    | Handcrafted       |         7.71 | 32-D, dicionário 100                  |
| **SIFT**                    | **Handcrafted**   |    **22.02** | **128-D, dicionário 200**             |
| ORB-EUC                     | Handcrafted       |        17.94 | 32-D, dicionário 200                  |
| ORB-HMM                     | Handcrafted       |        12.62 | 32-D, dicionário 200                  |
| **ResNet-18 (Fine-tuning)** | **Deep Learning** |    **96** | Pré-treinada no ImageNet, fine-tuning |
